In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

# =========================================================
# LOAD DATASET
# =========================================================

# Upload the Indian Climate Dataset CSV to Colab first
df = pd.read_csv('/content/indian_climate_dataset.csv')

print("FIRST 5 ROWS")
print(df.head())

print("\nDATASET SHAPE")
print(df.shape)

print("\nCOLUMN NAMES")
print(df.columns.tolist())

print("\nDATA TYPES")
print(df.dtypes)

print("\nDATASET INFORMATION")
df.info()

print("\nSTATISTICAL SUMMARY")
print(df.describe())

# =========================================================
# MISSING VALUES
# =========================================================

print("\nMISSING VALUES")
print(df.isnull().sum())

plt.figure(figsize=(12, 5))
sns.heatmap(df.isnull(), cbar=False, yticklabels=False)
plt.title("Missing Values Heatmap")
plt.xlabel("Features")
plt.ylabel("Rows")
plt.show()

# =========================================================
# DUPLICATES
# =========================================================

print("\nDUPLICATE ROWS")
print(df.duplicated().sum())

# =========================================================
# DATASET COLUMNS USED FOR CLIMATE EDA
# =========================================================

climate_features = [
    'Maximum Temperature',
    'Minimum Temperature',
    'Average Temperature',
    'Humidity',
    'Rainfall',
    'Wind Speed',
    'Pressure',
    'Cloud Cover',
    'AQI'
]

print("\nCLIMATE FEATURES USED FOR EDA")
print(climate_features)

# =========================================================
# CATEGORICAL DATA OVERVIEW
# =========================================================

categorical_columns = [
    col for col in ['City', 'State', 'AQI Category']
    if col in df.columns
]

for col in categorical_columns:
    print(f"\n{col.upper()} DISTRIBUTION")
    print(df[col].value_counts().head(15))

# =========================================================
# CITY DISTRIBUTION
# =========================================================

if 'City' in df.columns:
    plt.figure(figsize=(12, 6))
    city_counts = df['City'].value_counts().head(15)
    sns.barplot(x=city_counts.index, y=city_counts.values)
    plt.title("Top 15 Cities by Number of Climate Records")
    plt.xlabel("City")
    plt.ylabel("Number of Records")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

# =========================================================
# FEATURE DISTRIBUTIONS
# =========================================================

available_features = [feature for feature in climate_features if feature in df.columns]

n_features = len(available_features)
n_cols = 3
n_rows = int(np.ceil(n_features / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 5 * n_rows))
axes = np.array(axes).reshape(-1)

for i, feature in enumerate(available_features):
    sns.histplot(df[feature], bins=30, kde=True, ax=axes[i])
    axes[i].set_title(feature)
    axes[i].set_xlabel(feature)
    axes[i].set_ylabel("Frequency")

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Distribution of Indian Climate Features", fontsize=16)
plt.tight_layout()
plt.show()

# =========================================================
# BOX PLOTS / OUTLIER DETECTION
# =========================================================

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 5 * n_rows))
axes = np.array(axes).reshape(-1)

for i, feature in enumerate(available_features):
    sns.boxplot(y=df[feature], ax=axes[i])
    axes[i].set_title(feature)
    axes[i].set_ylabel(feature)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Boxplots of Indian Climate Features", fontsize=16)
plt.tight_layout()
plt.show()

# =========================================================
# NUMERICAL SUMMARY
# =========================================================

print("\nNUMERICAL FEATURES SUMMARY")
print(df[available_features].describe().T)

# =========================================================
# CORRELATION ANALYSIS
# =========================================================

correlation = df[available_features].corr()

plt.figure(figsize=(12, 9))
sns.heatmap(correlation, annot=True, cmap='coolwarm', fmt='.2f')
plt.title("Correlation Heatmap of Climate Features")
plt.tight_layout()
plt.show()

print("\nCORRELATION MATRIX")
print(correlation)

# =========================================================
# TEMPERATURE RELATIONSHIPS
# =========================================================

if 'Average Temperature' in df.columns and 'Humidity' in df.columns:
    plt.figure(figsize=(8, 6))
    sns.scatterplot(
        x='Average Temperature',
        y='Humidity',
        data=df,
        alpha=0.5
    )
    plt.title("Average Temperature vs Humidity")
    plt.xlabel("Average Temperature")
    plt.ylabel("Humidity")
    plt.tight_layout()
    plt.show()

if 'Average Temperature' in df.columns and 'Rainfall' in df.columns:
    plt.figure(figsize=(8, 6))
    sns.scatterplot(
        x='Average Temperature',
        y='Rainfall',
        data=df,
        alpha=0.5
    )
    plt.title("Average Temperature vs Rainfall")
    plt.xlabel("Average Temperature")
    plt.ylabel("Rainfall")
    plt.tight_layout()
    plt.show()

# =========================================================
# DATE-BASED ANALYSIS
# =========================================================

if 'Date' in df.columns:
    df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

    print("\nDATE RANGE")
    print("Start Date:", df['Date'].min())
    print("End Date:", df['Date'].max())

    monthly_temp = (
        df.dropna(subset=['Date'])
          .groupby(df['Date'].dt.to_period('M'))['Average Temperature']
          .mean()
          .reset_index()
    )

    monthly_temp['Date'] = monthly_temp['Date'].astype(str)

    if len(monthly_temp) > 0:
        plt.figure(figsize=(14, 6))
        sns.lineplot(
            x='Date',
            y='Average Temperature',
            data=monthly_temp,
            marker='o'
        )
        plt.title("Monthly Average Temperature")
        plt.xlabel("Month")
        plt.ylabel("Average Temperature")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

# =========================================================
# STATE-LEVEL CLIMATE OVERVIEW
# =========================================================

if 'State' in df.columns and 'Average Temperature' in df.columns:
    state_temperature = (
        df.groupby('State')['Average Temperature']
          .mean()
          .sort_values(ascending=False)
          .head(15)
    )

    plt.figure(figsize=(12, 6))
    sns.barplot(
        x=state_temperature.values,
        y=state_temperature.index
    )
    plt.title("Top 15 States by Average Temperature")
    plt.xlabel("Average Temperature")
    plt.ylabel("State")
    plt.tight_layout()
    plt.show()

# =========================================================
# CLIMATE FEATURE PREPARATION FOR CLUSTERING
# =========================================================

print("\nCLIMATE FEATURE PREPARATION")

clustering_features = [
    'Maximum Temperature',
    'Minimum Temperature',
    'Average Temperature',
    'Humidity',
    'Rainfall',
    'Wind Speed',
    'Pressure',
    'Cloud Cover'
]

clustering_features = [
    feature for feature in clustering_features
    if feature in df.columns
]

X_climate = df[clustering_features].copy()

print("Features selected for clustering:")
print(clustering_features)

print("\nMissing values in selected features:")
print(X_climate.isnull().sum())

print("\nEDA OBSERVATIONS")
print("1. Climate variables have different numerical ranges.")
print("2. Missing-value patterns should be checked before clustering.")
print("3. Boxplots help identify extreme climate observations.")
print("4. Correlation analysis helps identify relationships and possible redundancy.")
print("5. Standardization is required before distance-based clustering.")
print("6. No predefined target variable is required because the project uses unsupervised learning.")
print("7. The selected numerical climate features can be used as input for K-Means and Hierarchical Clustering.")

print("\nEDA COMPLETED SUCCESSFULLY!")


In [ ]:
# Optional: save the cleaned EDA-ready feature data for the clustering stage

# X_climate.to_csv('/content/climate_features_for_clustering.csv', index=False)

print("Ready for the next stage: Standardization -> K-Means -> Hierarchical Clustering")
